# Segmentasi Wilayah Pelanggan E-Commerce Shopee Berdasarkan Pola Pesanan, Pembayaran, dan Pengiriman Menggunakan K-Means dan Hierarchical Clustering

Proyek Akhir Praktikum Data Science, Prodi Informatika UPN "Veteran" Yogyakarta

Kelompok: `[isi nama kelompok]`
Anggota: `[isi nama dan NIM]`

## A. Business Understanding

### Masalah
Sebuah toko di Shopee melayani ratusan kota/kabupaten, tetapi kebijakan ongkir, promo, dan metode pembayaran cenderung diperlakukan sama untuk semua wilayah. Padahal pola belanja tiap wilayah bisa berbeda, misalnya nilai transaksi, ongkir, preferensi COD, dan tingkat pembatalan.

### Pertanyaan yang ingin dijawab
1. Berapa segmen wilayah yang terbentuk secara alami dari data pesanan?
2. Apa ciri tiap segmen (nilai transaksi, ongkir, berat, proporsi COD, tingkat pembatalan)?
3. Mana yang lebih baik dalam mengelompokkan wilayah: K-Means atau Hierarchical Clustering?
4. Strategi apa yang cocok untuk tiap segmen?

### Metode
Clustering (unsupervised) dengan **K-Means** dan **Agglomerative/Hierarchical Clustering** (linkage ward, average, complete, single) sebagai pembanding. Unit analisis adalah wilayah (kota/kabupaten), sehingga data pesanan akan diagregasi per wilayah.

### Metrik
- **Inertia/WCSS (Elbow)** dan **Silhouette Score** untuk memilih jumlah cluster dan membandingkan model.
- **Dendrogram** dan **Cophenetic Correlation Coefficient** untuk menilai struktur hierarki.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## B. Data Gathering

### Memuat data
File menggunakan pemisah `;` dan memiliki BOM di awal, sehingga dibaca dengan `sep=';'` dan `encoding='utf-8-sig'`.

In [ ]:
df = pd.read_csv("data/all_months_clean.csv", sep=";", encoding="utf-8-sig")
print("Dimensi data (baris, kolom):", df.shape)
df.head()

In [ ]:
df.info()

### Sumber dan lisensi
- Sumber: Kaggle, `[isi judul dataset]`
- Tautan: `[isi URL Kaggle]`
- Pembuat dataset: `[isi nama]`
- Lisensi: `[isi lisensi dari halaman Kaggle]`
- Tanggal pengambilan: `[isi tanggal unduh]`

### Pengertian dataset
Dataset berisi pesanan dari **satu toko di Shopee**, dengan periode `[isi periode sebenarnya dari data, lihat sel EDA]`. Setiap baris adalah satu pesanan. Dataset tidak memuat identitas pembeli (nama atau alamat lengkap), hanya kota/kabupaten dan provinsi. Dataset ini dipilih karena `[isi alasan kelompok]`.

Karena hanya mewakili satu toko di satu platform, hasil analisis tidak dapat digeneralisasi ke seluruh pasar e-commerce Indonesia.

### Arti kolom
*(Cocokkan dengan deskripsi kolom di halaman Kaggle, lalu perbaiki bila berbeda.)*

| Kolom | Arti |
|---|---|
| `order_id` | ID pesanan (identitas baris, bukan fitur) |
| `total_qty` | Total kuantitas produk dalam pesanan |
| `total_weight_gr` | Total berat pesanan (gram) |
| `total_returned_qty` | Kuantitas produk yang dikembalikan |
| `Total Diskon` | Total diskon pada pesanan |
| `product_categories` | Kategori produk dalam pesanan (bisa gabungan) |
| `num_product_categories` | Jumlah kategori produk dalam pesanan |
| `Status Pesanan` | Status akhir pesanan (selesai, batal, dikirim, dll.) |
| `Alasan Pembatalan` | Alasan pembatalan (hanya terisi untuk pesanan batal) |
| `Opsi Pengiriman` | Layanan kurir yang dipilih |
| `Metode Pembayaran` | Cara pembayaran (COD, ShopeePay, dll.) |
| `Kota/Kabupaten` | Wilayah tujuan pengiriman |
| `Provinsi` | Provinsi tujuan |
| `Ongkos Kirim Dibayar oleh Pembeli` | Ongkir yang dibayar pembeli |
| `Estimasi Potongan Biaya Pengiriman` | Estimasi potongan biaya kirim |
| `Total Pembayaran` | Total yang dibayar pembeli |
| `Perkiraan Ongkos Kirim` | Perkiraan ongkos kirim dari platform |
| `Waktu Pesanan Dibuat` | Waktu pesanan dibuat |
| `source_file` | Nama file ekspor bulanan asal baris (metadata, bukan fitur) |

## C. Exploratory Data Analysis (EDA)

### C.1 Statistik deskriptif

In [ ]:
df.describe().T

**Temuan:** `[isi dengan kata-kata sendiri setelah melihat output: rentang nilai, selisih mean dan median, tanda adanya outlier]`

### C.2 Missing value dan duplikat

In [ ]:
na = df.isna().sum()
na_pct = (na / len(df) * 100).round(2)
display(pd.DataFrame({"jumlah_kosong": na, "persen": na_pct})[na > 0])
print("Baris duplikat penuh:", df.duplicated().sum())
print("order_id duplikat:", df["order_id"].duplicated().sum())

**Temuan:** `[isi: kolom mana yang kosong, apakah kosong secara struktural (Alasan Pembatalan) atau karena data hilang (Waktu Pesanan Dibuat), dan rencana penanganannya di bagian D]`

### C.3 Periode data

In [ ]:
df["waktu"] = pd.to_datetime(df["Waktu Pesanan Dibuat"], errors="coerce")
print("Awal:", df["waktu"].min(), "| Akhir:", df["waktu"].max())
print("Waktu tidak terbaca:", df["waktu"].isna().sum())

per_bulan = df.groupby(df["waktu"].dt.to_period("M")).size()
plt.figure(figsize=(12, 4))
per_bulan.plot(kind="bar")
plt.title("Jumlah pesanan per bulan")
plt.xlabel("Bulan"); plt.ylabel("Jumlah pesanan")
plt.tight_layout(); plt.show()

**Temuan:** `[isi: periode sebenarnya, bulan dengan pesanan tertinggi, bulan yang hilang atau kosong]`

### C.4 Status pesanan, pembayaran, dan pengiriman

In [ ]:
df["status_ringkas"] = df["Status Pesanan"].str.split(",").str[0].str.strip()
display(df["status_ringkas"].value_counts())
display(df["Metode Pembayaran"].value_counts())

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
df["status_ringkas"].value_counts().head(6).plot(kind="barh", ax=ax[0], title="Status pesanan")
df["Metode Pembayaran"].value_counts().head(8).plot(kind="barh", ax=ax[1], title="Metode pembayaran")
plt.tight_layout(); plt.show()

**Temuan:** `[isi: proporsi pesanan batal, dominasi COD, label status yang tidak konsisten]`

In [ ]:
df["kurir"] = (df["Opsi Pengiriman"].str.upper()
               .str.extract(r"(SPX|JNE|J&T|GOSEND|GRABEXPRESS)", expand=False))
display(df["kurir"].value_counts(dropna=False))
display(df["Opsi Pengiriman"].nunique())

**Temuan:** `[isi: kurir dominan, dan bahwa Opsi Pengiriman punya banyak label ganda untuk layanan yang sama]`

### C.5 Sebaran wilayah

In [ ]:
print("Jumlah kota/kabupaten:", df["Kota/Kabupaten"].nunique())
print("Jumlah provinsi:", df["Provinsi"].nunique())
per_wilayah = df["Kota/Kabupaten"].value_counts()
display(per_wilayah.head(10))

plt.figure(figsize=(10, 4))
sns.histplot(per_wilayah, bins=50, log_scale=(False, True))
plt.title("Sebaran jumlah pesanan per wilayah")
plt.xlabel("Pesanan per wilayah"); plt.ylabel("Jumlah wilayah (skala log)")
plt.tight_layout(); plt.show()

for t in [5, 10, 20, 30, 50]:
    s = per_wilayah[per_wilayah >= t]
    print(f"Minimal {t:>2} pesanan -> {len(s):>3} wilayah, mencakup {s.sum()/len(df)*100:.1f}% pesanan")

**Temuan:** `[isi: distribusi sangat timpang, dan alasan memilih batas minimal pesanan per wilayah untuk bagian D]`

### C.6 Outlier pada fitur numerik

In [ ]:
num_cols = ["total_qty", "total_weight_gr", "Total Diskon", "Total Pembayaran", "Perkiraan Ongkos Kirim"]
fig, axes = plt.subplots(1, len(num_cols), figsize=(16, 4))
for ax, c in zip(axes, num_cols):
    sns.boxplot(y=df[c], ax=ax)
    ax.set_title(c)
plt.tight_layout(); plt.show()

for c in num_cols:
    q1, q3 = df[c].quantile([0.25, 0.75]); iqr = q3 - q1
    n = ((df[c] < q1 - 1.5*iqr) | (df[c] > q3 + 1.5*iqr)).sum()
    print(f"{c:<26} outlier IQR: {n} ({n/len(df)*100:.1f}%)")

**Temuan:** `[isi: kolom mana yang paling banyak outlier dan apa artinya bagi K-Means yang sensitif terhadap outlier]`

### C.7 Kebocoran dan keterkaitan antarvariabel

In [ ]:
display(df.groupby("status_ringkas")["Total Pembayaran"].describe().round(0))

plt.figure(figsize=(8, 6))
sns.heatmap(df.select_dtypes("number").corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Korelasi fitur numerik")
plt.tight_layout(); plt.show()

**Temuan:** `[isi: nilai Total Pembayaran pada pesanan batal, fitur yang berkorelasi tinggi, dan implikasinya pada pemilihan fitur]`

## D. Cleaning dan Preprocessing

*(Isi setelah EDA selesai. Tampilkan kondisi data **sebelum dan sesudah** tiap langkah.)*

- [ ] D.1 Penanganan duplikat dan missing value
- [ ] D.2 Konversi tipe data (waktu) dan penyeragaman `Status Pesanan` dan `Opsi Pengiriman`
- [ ] D.3 Keputusan status yang dikeluarkan ("Sedang Dikirim", "Telah Dikirim")
- [ ] D.4 Agregasi per wilayah (`groupby('Kota/Kabupaten').agg(...)`) dan batas minimal pesanan
- [ ] D.5 Penanganan outlier dan skewness (IQR, transformasi log)
- [ ] D.6 Feature selection: fitur yang dipakai, dibuang, dan dibuat, beserta alasannya
- [ ] D.7 Scaling dengan `StandardScaler`

## E. Modelling

- [ ] E.1 Elbow Method dan Silhouette Score untuk K = 2 sampai 10
- [ ] E.2 K-Means (`n_init=10`, `random_state=42`)
- [ ] E.3 Agglomerative Clustering dengan linkage ward, average, complete, single
- [ ] E.4 Dendrogram (ward) dan penentuan jumlah cluster dari lompatan tinggi merge

## F. Evaluasi

- [ ] F.1 Tabel Silhouette Score semua model
- [ ] F.2 Cophenetic Correlation Coefficient tiap linkage
- [ ] F.3 Heatmap z-score fitur yang diurutkan per cluster
- [ ] F.4 Profil tiap cluster (`groupby`, `crosstab`)
- [ ] F.5 Perbandingan K-Means vs Hierarchical dan alasan memilih model terbaik

## G. Konklusi

- [ ] Jawaban atas empat pertanyaan di Business Understanding
- [ ] Penamaan tiap segmen wilayah dan terjemahan temuan ke bahasa bisnis
- [ ] Rekomendasi strategi per segmen
- [ ] Keputusan yang diambil
- [ ] Keterbatasan (satu toko, satu platform, jumlah wilayah, outlier)